<a href="https://colab.research.google.com/github/dhanush190312/MenuManagerip/blob/main/MenuManager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [ ]:
!pip install transformers datasets sentence-transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/menu.csv')
print(df.shape)
df.head()


(117, 5)


,dish_name,category,price,description,dietary_tags
0,Paneer Tikka,Starter,260,Char-grilled cubes of fresh paneer marinated o...,"Veg, Spicy, Contains Dairy"
1,Paneer 65,Starter,240,Crispy fried paneer cubes tossed in a fiery So...,"Veg, Spicy, Contains Dairy"
2,Paneer Chilli Dry,Starter,250,Golden fried paneer cubes wok-tossed with caps...,"Veg, Spicy, Contains Dairy"
3,Veg Manchurian Dry,Starter,220,Crispy vegetable and cabbage dumplings tossed ...,"Veg, Spicy"
4,Gobi Manchurian,Starter,220,Crispy battered cauliflower florets tossed in ...,"Veg, Spicy"


In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Candidate labels from your dataset
candidate_labels = df["category"].unique().tolist()

# Example dish
dish = "Paneer Butter Masala"

result = classifier(
    dish,
    candidate_labels=candidate_labels
)

print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'Paneer Butter Masala', 'labels': ['Main Course', 'Soup', 'Starter', 'Dessert', 'Beverage', 'Biryani', 'Bread', 'Rice', 'Salad'], 'scores': [0.3711334764957428, 0.16100694239139557, 0.15396343171596527, 0.069997638463974, 0.06798302382230759, 0.05318520590662956, 0.04499925673007965, 0.04245685413479805, 0.03527418151497841]}


In [ ]:
correct = 0
wrong = []

for _, row in df.iterrows():
    prediction = classifier(
        row["dish_name"],
        candidate_labels=candidate_labels,
        hypothesis_template="This menu item falls under the {} category."
    )
    predicted = prediction["labels"][0].strip().lower()
    actual = row["category"].strip().lower()

    if predicted == actual:
        correct += 1
    else:
        wrong.append({
            "dish": row["dish_name"],
            "actual": row["category"],
            "predicted": prediction["labels"][0]
        })

accuracy = correct / len(df)
print(f"Total dishes  : {len(df)}")
print(f"Correct       : {correct}")
print(f"Wrong         : {len(wrong)}")
print(f"Accuracy      : {accuracy:.2%}")
print("\n--- Mismatches ---")
for w in wrong:
    print(f"  {w['dish']:<30} | Actual: {w['actual']:<12} | Predicted: {w['predicted']}")

Total dishes  : 117
Correct       : 68
Wrong         : 49
Accuracy      : 58.12%

--- Mismatches ---
  Paneer 65                      | Actual: Starter      | Predicted: Soup
  Paneer Chilli Dry              | Actual: Starter      | Predicted: Beverage
  Veg Manchurian Dry             | Actual: Starter      | Predicted: Beverage
  Gobi Manchurian                | Actual: Starter      | Predicted: Main Course
  Hara Bhara Kabab               | Actual: Starter      | Predicted: Biryani
  Veg Seekh Kabab                | Actual: Starter      | Predicted: Main Course
  Paneer Malai Tikka             | Actual: Starter      | Predicted: Main Course
  Chicken 65                     | Actual: Starter      | Predicted: Main Course
  Chicken Lollipop               | Actual: Starter      | Predicted: Dessert
  Tandoori Chicken Full          | Actual: Starter      | Predicted: Main Course
  Fish Amritsari                 | Actual: Starter      | Predicted: Main Course
  Fish Tikka                 

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_description(dish_name, category, dietary_tags=""):
    tags_hint = f" It is {dietary_tags}." if dietary_tags else ""
    prompt = (
        f"You are a professional restaurant menu copywriter. "
        f"Write exactly one appetizing sentence (15-25 words) "
        f"describing '{dish_name}', a {category.lower()} item.{tags_hint} "
        f"Mention the cooking style and key flavors. "
        f"Do not start with the dish name. Do not repeat the prompt."
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.75,
        top_p=0.92,
        repetition_penalty=1.4
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if len(text.split()) < 8:
        text = (
            f"A carefully prepared {category.lower()} crafted with "
            f"premium ingredients and authentic flavors."
        )
    return text

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
test_cases = [
    ("Chicken Biryani",    "Biryani",     "Non-Veg, Spicy"),
    ("Paneer Tikka",       "Starter",     "Veg, Spicy, Contains Dairy"),
    ("Gulab Jamun",        "Dessert",     "Veg, Contains Dairy"),
    ("Garlic Naan",        "Bread",       "Veg, Contains Dairy"),
    ("Mango Lassi",        "Beverage",    "Veg, Contains Dairy"),
]

print(f"{'Dish':<25} {'Category':<12} Generated Description")
print("-" * 85)
for dish, cat, tags in test_cases:
    desc = generate_description(dish, cat, tags)
    print(f"{dish:<25} {cat:<12} {desc}")

Dish                      Category     Generated Description
-------------------------------------------------------------------------------------
Chicken Biryani           Biryani      Chicken Biryani is a biryani item that is spicy.
Paneer Tikka              Starter      A carefully prepared starter crafted with premium ingredients and authentic flavors.
Gulab Jamun               Dessert      Gulab Jamun is a Veg, Contains Dairy dessert.
Garlic Naan               Bread        Garlic Naan is a bread item that contains vegetables.
Mango Lassi               Beverage     A carefully prepared beverage crafted with premium ingredients and authentic flavors.


In [ ]:
def predict_category(dish_name):
    result = classifier(
        dish_name,
        candidate_labels=candidate_labels,
        hypothesis_template="This menu item falls under the {} category."
    )
    return result["labels"][0], round(result["scores"][0], 2)

def menu_assistant(dish_name):
    category, confidence = predict_category(dish_name)

    match = df[df["dish_name"].str.lower() == dish_name.lower()]
    tags = match["dietary_tags"].values[0] if len(match) > 0 else ""

    description = generate_description(dish_name, category, tags)

    return {
        "dish_name"            : dish_name,
        "predicted_category"   : category,
        "confidence"           : confidence,
        "generated_description": description
    }

In [ ]:
print("=" * 55)
print("       MenuManager — Smart Menu Assistant")
print("=" * 55)

while True:
    dish = input("\n Enter dish name (or 'quit' to exit): ").strip()
    if dish.lower() in ("quit", "exit", ""):
        print("\n Exiting MenuManager. Goodbye!")
        break
    result = menu_assistant(dish)
    print(f"\n  Dish       : {result['dish_name']}")
    print(f"  Category   : {result['predicted_category']}  (confidence: {result['confidence']})")
    print(f"  Description: {result['generated_description']}")
    print("-" * 55)

       MenuManager — Smart Menu Assistant

 Enter dish name (or 'quit' to exit): chiken briyani

  Dish       : chiken briyani
  Category   : Biryani  (confidence: 0.78)
  Description: Chin briyani is an Indian dish, and is often served as an accompaniment to other dishes.
-------------------------------------------------------

 Enter dish name (or 'quit' to exit): water

  Dish       : water
  Category   : Beverage  (confidence: 0.77)
  Description: The water is a beverage that you can enjoy in your own home.
-------------------------------------------------------

 Enter dish name (or 'quit' to exit): veg fried rice

  Dish       : veg fried rice
  Category   : Rice  (confidence: 0.88)
  Description: Veg fried rice is a dish that can be eaten as a main course, or as an entree.
-------------------------------------------------------

 Enter dish name (or 'quit' to exit): exit

 Exiting MenuManager. Goodbye!


In [ ]:
def menu_assistant_safe(dish_name, threshold=0.45):
    category, confidence = predict_category(dish_name)

    if confidence < threshold:
        return {
            "dish_name": dish_name,
            "predicted_category": "Uncertain — please choose manually",
            "confidence": round(confidence, 2),
            "generated_description": "N/A"
        }

    match = df[df["dish_name"].str.lower() == dish_name.lower()]
    tags = match["dietary_tags"].values[0] if len(match) > 0 else ""
    description = generate_description(dish_name, category, tags)

    return {
        "dish_name": dish_name,
        "predicted_category": category,
        "confidence": round(confidence, 2),
        "generated_description": description
    }

# Test with ambiguous dish
print(menu_assistant_safe("veg fried rice"))
print(menu_assistant_safe("Chicken Biryani"))

{'dish_name': 'veg fried rice', 'predicted_category': 'Rice', 'confidence': 0.88, 'generated_description': 'Veg fried rice is a dish that is veg and a veg vegetable.'}
{'dish_name': 'Chicken Biryani', 'predicted_category': 'Biryani', 'confidence': 0.97, 'generated_description': 'The chicken biryani is a dish that has a spicy, non-vegetarian, and non-veg.'}


gpt2

In [ ]:
from transformers import pipeline

gpt2 = pipeline(
    "text-generation",
    model="gpt2"
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
prompt = "Restaurant description for Chicken Biryani."

result = gpt2(
    prompt,
    max_new_tokens=25
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=T

Restaurant description for Chicken Biryani.

- Filled with fresh, fresh ingredients.

- Vegan friendly and gluten-free.

- All


In [ ]:
while True:

    dish = input("Enter Dish Name (or exit): ")

    if dish.lower() == "exit":
        break

    category, confidence = predict_category(dish)

    description = generate_description(
        dish,
        category,
        ""
    )

    print("\nCategory :", category)
    print("Confidence :", round(confidence,2))
    print("Description :", description)

Enter Dish Name (or exit): chicken briyani

Category : Biryani
Confidence : 0.88
Description : A carefully prepared biryani crafted with premium ingredients and authentic flavors.
Enter Dish Name (or exit): veg frieed rice

Category : Rice
Confidence : 0.9
Description : Veg frieed rice is a dish that has a lot of flavor.
Enter Dish Name (or exit): exit


In [ ]:
def menu_assistant(dish_name, dietary_tags=""):

    category, confidence = predict_category(dish_name)

    description = generate_description(
        dish_name,
        category,
        dietary_tags
    )

    return {
        "Dish": dish_name,
        "Predicted Category": category,
        "Confidence": confidence,
        "Generated Description": description
    }

print(menu_assistant("Chicken Biryani"))

{'Dish': 'Chicken Biryani', 'Predicted Category': 'Biryani', 'Confidence': 0.97, 'Generated Description': 'Chicken Biryani is a biryani dish that is prepared with chicken, onions, and curry.'}
